In [1]:
"""
UWF-ZeekData24 Intrusion Detection Pipeline.

Stages:
    1. Data acquisition
    2. Data loading
    3. Data audit
    4. Leak-column removal (identifiers + pre-existing label columns)
    5. Train / validation / test split (stratified)
    6. Preprocessing (categorical encoding, imputation, target encoding, scaling)
    7. Model definitions
    8. Model evaluation (binary and multi-class), all output printed
"""

import os
import re
import glob
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =========================================================
# Configuration
# =========================================================
DATASET_BASE_URL = "https://datasets.uwf.edu/data/UWF-ZeekData24/csv/"
DATA_DIR = "data/UWF-ZeekData24/csv"

ATTACK_CATEGORIES = [
    "Benign",
    "Credential_Access",
    "Defense_Evasion",
    "Exfiltration",
    "Initial_Access",
    "Persistence",
    "Privilege_Escalation",
    "Reconnaissance",
]

RANDOM_STATE = 42
TRAIN_FRACTION = 0.6
VAL_FRACTION = 0.2  # remainder (0.2) is used for the test set

LABEL_COLUMN = "tactic_label"
BENIGN_LABEL = "Benign"

# ---------------------------------------------------------
# Leak-prone columns, identified by inspecting the ACTUAL column list of
# this dataset run (`df.columns.tolist()`), not assumed from generic Zeek
# naming conventions. An earlier version used substring-pattern matching
# (e.g. looking for "id.orig_h") which silently failed here because this
# dataset uses different names (src_ip_zeek, dest_ip_zeek, ...), and
# separately caused a false positive by matching "ts" inside "orig_pkts".
# Explicit, exact column names avoid both failure modes.
# ---------------------------------------------------------

# Per-connection identifiers: a model can memorize "this IP/port always
# means this attack" instead of learning transferable behavior.
IDENTIFIER_COLUMNS = [
    "uid",  # unique per-connection ID
    "ts",  # unix timestamp of connection start
    "datetime",  # human-readable timestamp -- same info as ts
    "src_ip_zeek",  # source (attacker) IP
    "src_port_zeek",  # source port
    "dest_ip_zeek",  # destination (victim) IP
    "dest_port_zeek",  # destination port
    "community_id",  # hash of the (IP, port, proto) 5-tuple -- encodes
    # the same identifying info as the IP/port columns
    # above, just hashed into a single string
]

# Columns that ARE the label, at various levels of granularity. tactic_label
# is the target WE assign (from the folder name in load_dataset) and is
# excluded separately via LABEL_COLUMN. These four are the dataset's OWN
# pre-existing label columns, which must never be used as features --
# label_binary in particular is literally the binary target itself.
LABEL_LEAK_COLUMNS = [
    "label_tactic",
    "label_technique",
    "label_binary",
    "label_cve",
]


# =========================================================
# 1. Data Acquisition
# =========================================================
def list_remote_csv_files(category_url: str) -> list[str]:
    """Return the absolute URLs of all CSV files listed on a category index page."""
    response = requests.get(category_url, timeout=30)
    response.raise_for_status()
    relative_links = re.findall(r'href="([^"]+\.csv)"', response.text)
    return [urljoin(category_url, link) for link in relative_links]


def download_file(url: str, destination: str) -> None:
    """Download a single file to `destination`, skipping it if it already exists."""
    if os.path.exists(destination):
        print(f"[skip] already downloaded: {destination}")
        return

    os.makedirs(os.path.dirname(destination), exist_ok=True)
    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with open(destination, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
    print(f"[done] downloaded: {destination}")


def download_dataset(base_url: str, data_dir: str, categories: list[str]) -> None:
    """Download every CSV file for every category into `data_dir/<category>/`."""
    for category in categories:
        category_url = urljoin(base_url, category + "/")
        try:
            csv_urls = list_remote_csv_files(category_url)
        except requests.RequestException as e:
            print(f"[error] could not list files for {category}: {e}")
            continue

        if not csv_urls:
            print(f"[warning] no CSV files found for category: {category}")
            continue

        for csv_url in csv_urls:
            filename = os.path.basename(csv_url)
            destination = os.path.join(data_dir, category, filename)
            download_file(csv_url, destination)


# =========================================================
# 2. Data Loading
# =========================================================
def load_dataset(
    data_dir: str, categories: list[str], label_column: str
) -> pd.DataFrame:
    """Load and concatenate all CSV files, labeling each row by its category folder."""
    frames = []
    for category in categories:
        category_dir = os.path.join(data_dir, category)
        for csv_path in glob.glob(os.path.join(category_dir, "*.csv")):
            frame = pd.read_csv(csv_path, low_memory=False)
            frame[label_column] = category
            frames.append(frame)
            print(f"Loaded {csv_path}: {frame.shape[0]} rows")

    if not frames:
        raise FileNotFoundError(f"No CSV files found under {data_dir}")

    return pd.concat(frames, ignore_index=True)


# =========================================================
# 3. Data Audit
# =========================================================
def audit_dataset(data: pd.DataFrame, label_column: str) -> None:
    """Report missing values, duplicates, and class balance without modifying the data."""
    missing_by_column = data.isnull().sum()
    print("Missing values per column:")
    print(missing_by_column[missing_by_column > 0])

    print(f"\nDuplicate rows: {data.duplicated().sum()}")

    print("\nClass distribution (%):")
    print((data[label_column].value_counts(normalize=True) * 100).round(2))


# =========================================================
# 4. Leak-Column Removal
# =========================================================
def drop_known_leak_columns(
    data: pd.DataFrame, columns_to_drop: list[str]
) -> pd.DataFrame:
    """Drop an explicit list of columns, reporting exactly what happened.

    Uses exact-name matching only (no substring patterns), so it can't
    silently over-match (e.g. "ts" inside "orig_pkts") or silently
    under-match (e.g. assuming id.orig_h when the real column is
    src_ip_zeek). Anything expected-but-missing is reported so a naming
    mismatch is caught immediately instead of causing silent leakage.
    """
    present = [c for c in columns_to_drop if c in data.columns]
    missing = [c for c in columns_to_drop if c not in data.columns]

    print(f"Dropping leak-prone columns ({len(present)}): {present}")
    if missing:
        print(
            f"[note] expected leak columns NOT found in this dataset "
            f"(check naming): {missing}"
        )

    return data.drop(columns=present)


# =========================================================
# 5. Train / Validation / Test Split
# =========================================================
def stratified_split(
    data: pd.DataFrame,
    label_column: str,
    train_fraction: float,
    val_fraction: float,
    random_state: int,
):
    """Stratified split — every class is proportionally represented in
    train, validation, and test.

    Used instead of a chronological split because UWF-ZeekData24 is a
    single fixed-duration capture: attack categories occupy narrow,
    non-overlapping time windows, unlike a genuine multi-year traffic
    archive. Real chronological/temporal-drift evaluation is instead done
    separately on MAWIFlow, where it is appropriate.
    """
    test_fraction = 1 - train_fraction
    train_data, remainder = train_test_split(
        data,
        test_size=test_fraction,
        stratify=data[label_column],
        random_state=random_state,
    )
    relative_val_fraction = val_fraction / test_fraction
    val_data, test_data = train_test_split(
        remainder,
        test_size=1 - relative_val_fraction,
        stratify=remainder[label_column],
        random_state=random_state,
    )
    return train_data, val_data, test_data


# =========================================================
# 6. Preprocessing
# =========================================================
def encode_categorical_features(
    train_data: pd.DataFrame, test_data: pd.DataFrame, exclude: list[str]
):
    """Label-encode categorical columns, fitting on the training set only.

    Categories seen only in the test set are mapped to -1.
    """
    train_data = train_data.copy()
    test_data = test_data.copy()

    categorical_columns = train_data.select_dtypes(
        include=["object", "str"]
    ).columns.difference(exclude)

    for column in categorical_columns:
        encoder = LabelEncoder()
        train_data[column] = encoder.fit_transform(train_data[column].astype(str))
        test_data[column] = (
            test_data[column]
            .astype(str)
            .map(
                lambda value, enc=encoder: (
                    enc.transform([value])[0] if value in enc.classes_ else -1
                )
            )
        )

    return train_data, test_data


def impute_missing_and_infinite(X_train: pd.DataFrame, X_test: pd.DataFrame):
    """Impute NaNs with the training median and cap infinities at the training max.

    Fit exclusively on X_train; X_test is only transformed.
    """
    X_train = X_train.copy()
    X_test = X_test.copy()

    numeric_columns = X_train.select_dtypes(include=["number"]).columns.tolist()

    missing_before = X_train[numeric_columns].isnull().sum()
    columns_with_missing = missing_before[missing_before > 0]
    if len(columns_with_missing) > 0:
        print("Columns with missing values (train set):")
        print(columns_with_missing)

    imputer = SimpleImputer(strategy="median")
    X_train[numeric_columns] = imputer.fit_transform(X_train[numeric_columns])
    X_test[numeric_columns] = imputer.transform(X_test[numeric_columns])

    infinite_counts = np.isinf(X_train[numeric_columns]).sum()
    columns_with_inf = infinite_counts[infinite_counts > 0]
    if len(columns_with_inf) > 0:
        print("\nColumns with infinite values (train set), capped at column max:")
        print(columns_with_inf)
        for column in columns_with_inf.index:
            finite_values = X_train.loc[X_train[column] != np.inf, column]
            column_max = finite_values.max()
            X_train[column] = X_train[column].replace(np.inf, column_max)
            X_test[column] = X_test[column].replace(np.inf, column_max)

    return X_train, X_test, numeric_columns


def scale_features(
    X_train: pd.DataFrame, X_test: pd.DataFrame, numeric_columns: list[str]
):
    """Standardize numeric features (zero mean, unit variance), fit on train only.

    Tree-based models are scale-invariant and do not need this, but the ANN
    trains poorly, or not at all, on raw unscaled features when columns span
    very different ranges (e.g. duration vs. byte counts).
    """
    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    scaler = StandardScaler()
    X_train_scaled[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
    X_test_scaled[numeric_columns] = scaler.transform(X_test[numeric_columns])

    return X_train_scaled, X_test_scaled


# =========================================================
# 7. Model Definitions
# =========================================================
def build_tree_based_models(random_state: int) -> dict:
    """Tree-based models, trained on unscaled features."""
    return {
        "DecisionTree": DecisionTreeClassifier(
            class_weight="balanced",
            min_samples_leaf=1,
            ccp_alpha=0.0,
            random_state=random_state,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200,
            max_features="sqrt",
            min_samples_leaf=1,
            ccp_alpha=0.0,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=200,
            min_samples_leaf=1,
            ccp_alpha=0.0,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "Bagging(DecisionTree)": BaggingClassifier(
            estimator=DecisionTreeClassifier(
                class_weight="balanced",
                min_samples_leaf=1,
                ccp_alpha=0.0,
                random_state=random_state,
            ),
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1,
        ),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=200, learning_rate=1.0, random_state=random_state
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.1,
            min_samples_leaf=1,
            ccp_alpha=0.0,
            random_state=random_state,
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, random_state=random_state
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, eval_metric="logloss", random_state=random_state
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=200, random_state=random_state, verbosity=-1
        ),
        "CatBoost": CatBoostClassifier(
            iterations=200, random_state=random_state, verbose=False
        ),
    }


def build_ann_model(random_state: int) -> dict:
    """Neural network model, trained separately on scaled features."""
    return {
        "ANN": MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=300,
            early_stopping=True,
            random_state=random_state,
        ),
    }


# =========================================================
# 8. Model Evaluation and Visualization
# =========================================================
def evaluate_models(
    models: dict,
    X_train,
    y_train,
    X_test,
    y_test,
    average_mode: str,
    labels=None,
    target_names=None,
):
    """Train and evaluate each model, returning a summary table, detailed
    reports, and each model's raw predictions (for plotting confusion matrices).
    """
    summary_rows = []
    detailed_reports = []
    predictions_by_model = {}

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)
        predictions_by_model[model_name] = predictions

        accuracy = accuracy_score(y_test, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, predictions, average=average_mode, labels=labels, zero_division=0
        )

        summary_rows.append(
            {
                "Model": model_name,
                "Accuracy": accuracy,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
            }
        )

        report = classification_report(
            y_test,
            predictions,
            labels=labels,
            target_names=target_names,
            zero_division=0,
        )
        detailed_reports.append(f"[{model_name}]\n{report}")
        print(f"{model_name} -- Accuracy={accuracy:.4f}, F1={f1:.4f}")

    summary = pd.DataFrame(summary_rows).sort_values("F1", ascending=False)
    return summary, detailed_reports, predictions_by_model


# =========================================================
# Pipeline execution (top-level, not wrapped in a function, so that df,
# X_train, models, etc. remain accessible afterward for interactive
# inspection and debugging)
# =========================================================

# --- Data acquisition and loading ---
download_dataset(DATASET_BASE_URL, DATA_DIR, ATTACK_CATEGORIES)
df = load_dataset(DATA_DIR, ATTACK_CATEGORIES, LABEL_COLUMN)
print(f"\nCombined dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# --- Data audit (on the raw, unmodified data) ---
audit_dataset(df, LABEL_COLUMN)

# --- Remove identifier + pre-existing label columns (leakage prevention) ---
# NOTE: explicit exact-name lists, not substring patterns -- see comments
# on IDENTIFIER_COLUMNS / LABEL_LEAK_COLUMNS above for why.
df = drop_known_leak_columns(df, IDENTIFIER_COLUMNS + LABEL_LEAK_COLUMNS)
print(
    f"Columns remaining after leak-column removal ({df.shape[1]}): {df.columns.tolist()}"
)

# --- Split (stratified, not chronological — see stratified_split docstring) ---
train_df, val_df, test_df = stratified_split(
    df, LABEL_COLUMN, TRAIN_FRACTION, VAL_FRACTION, RANDOM_STATE
)
print(f"\nTrain: {train_df.shape}, Validation: {val_df.shape}, Test: {test_df.shape}")

# --- Preprocessing ---
train_df, test_df = encode_categorical_features(
    train_df, test_df, exclude=[LABEL_COLUMN]
)

train_df["binary_label"] = (train_df[LABEL_COLUMN] != BENIGN_LABEL).astype(int)
test_df["binary_label"] = (test_df[LABEL_COLUMN] != BENIGN_LABEL).astype(int)

feature_columns = train_df.columns.difference([LABEL_COLUMN, "binary_label"])
print(
    f"\nFinal feature columns used for training ({len(feature_columns)}): "
    f"{list(feature_columns)}"
)

X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

y_train_binary, y_test_binary = train_df["binary_label"], test_df["binary_label"]
y_train_multiclass_raw = train_df[LABEL_COLUMN]
y_test_multiclass_raw = test_df[LABEL_COLUMN]

X_train, X_test, numeric_columns = impute_missing_and_infinite(X_train, X_test)

target_encoder = LabelEncoder()
y_train_multiclass = target_encoder.fit_transform(y_train_multiclass_raw)
y_test_multiclass = target_encoder.transform(y_test_multiclass_raw)
class_labels = list(range(len(target_encoder.classes_)))

print("\nMulti-class label mapping:")
for index, class_name in enumerate(target_encoder.classes_):
    print(f"  {index}: {class_name}")

# Scaled feature set, for the ANN only
X_train_scaled, X_test_scaled = scale_features(X_train, X_test, numeric_columns)

# --- Models ---
tree_models = build_tree_based_models(RANDOM_STATE)
ann_model = build_ann_model(RANDOM_STATE)

# --- Evaluation: binary task ---
print("\n" + "=" * 60)
print("BINARY CLASSIFICATION (Normal vs Attack)")
print("=" * 60)

tree_bin_summary, tree_bin_reports, tree_bin_preds = evaluate_models(
    tree_models, X_train, y_train_binary, X_test, y_test_binary, average_mode="binary"
)
ann_bin_summary, ann_bin_reports, ann_bin_preds = evaluate_models(
    ann_model,
    X_train_scaled,
    y_train_binary,
    X_test_scaled,
    y_test_binary,
    average_mode="binary",
)
binary_summary = pd.concat(
    [tree_bin_summary, ann_bin_summary], ignore_index=True
).sort_values("F1", ascending=False)
binary_predictions = {**tree_bin_preds, **ann_bin_preds}

print(binary_summary.to_string(index=False))

# --- Evaluation: multi-class task ---
print("\n" + "=" * 60)
print("MULTI-CLASS CLASSIFICATION (Tactic Identification)")
print("=" * 60)

tree_multi_summary, tree_multi_reports, tree_multi_preds = evaluate_models(
    tree_models,
    X_train,
    y_train_multiclass,
    X_test,
    y_test_multiclass,
    average_mode="macro",
    labels=class_labels,
    target_names=target_encoder.classes_,
)
ann_multi_summary, ann_multi_reports, ann_multi_preds = evaluate_models(
    ann_model,
    X_train_scaled,
    y_train_multiclass,
    X_test_scaled,
    y_test_multiclass,
    average_mode="macro",
    labels=class_labels,
    target_names=target_encoder.classes_,
)
multiclass_summary = pd.concat(
    [tree_multi_summary, ann_multi_summary], ignore_index=True
).sort_values("F1", ascending=False)
multiclass_predictions = {**tree_multi_preds, **ann_multi_preds}

print(multiclass_summary.to_string(index=False))

# --- Per-class detail (precision/recall/support per tactic, per model) ---
# Macro-F1 alone hides WHICH classes are failing and why (e.g. too few test
# samples vs. genuinely hard to distinguish). Print every model's full
# classification_report so that's visible before choosing an imbalance
# strategy (oversampling, merging rare classes, etc.).
print("\n" + "=" * 60)
print("PER-CLASS DETAIL (Multi-class)")
print("=" * 60)
for report_text in tree_multi_reports + ann_multi_reports:
    print(report_text)
    print("-" * 60)

[skip] already downloaded: data/UWF-ZeekData24/csv\Benign\part-00000-b039213f-1208-4f1b-b71d-5e8e3a4a2939-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Credential_Access\part-00000-912fdc44-5727-4d42-8fd6-a0e206ba29f8-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Defense_Evasion\part-00000-de4985c7-284a-4b6e-b507-e69a2d172582-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Exfiltration\part-00000-6a530c25-0f6b-46a1-ba16-c6b658ef75e8-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Initial_Access\part-00000-9a37b839-429e-444b-82a5-a6d5e69dad7e-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Persistence\part-00000-fb5a764a-e65e-4e10-b6fb-1a189589d5e0-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Privilege_Escalation\part-00000-d8fcf83a-aed1-4cb1-babf-581208088293-c000.csv
[skip] already downloaded: data/UWF-ZeekData24/csv\Reconnaissance\part-00000-49ed1cc6-3205-4c76-8e28-4da9abf78363-c000.csv
Loaded data/UWF-Zee